# MiVOLO v2 Fine-tuning on Indian Face Data — AgeVision

**Goal:** Fine-tune MiVOLO v2 age estimator on Indian faces (IFAD + UTKFace-Indian) for improved accuracy.

**MiVOLO v2 baseline:** ~3.65 MAE (LAGENDA), ~4.24 MAE (IMDB-Clean) — general population.  
**Target:** Lower MAE specifically on Indian/South Asian faces.

## Prerequisites
1. Open in Colab: **Runtime → Change runtime type → GPU (T4 or better)**
2. (Optional) Kaggle API key for UTKFace download, or upload UTKFace manually
3. Run all cells

In [ ]:
#@title Step 1: Install Dependencies & Verify GPU
!pip install -q torch torchvision transformers accelerate ultralytics huggingface_hub kagglehub
!pip install -q opencv-python-headless pillow tqdm matplotlib
!pip install -q mivolo

import os, re, csv, shutil, glob, time
from pathlib import Path

import cv2
import numpy as np
import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import Dataset, DataLoader, Subset, random_split
import torchvision.transforms as T

print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if not torch.cuda.is_available():
    raise RuntimeError('No GPU! Runtime -> Change runtime type -> GPU')

gpu = torch.cuda.get_device_name(0)
mem = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {gpu} ({mem:.1f} GB)')

if mem >= 40:
    REC_BATCH = 32
elif mem >= 15:
    REC_BATCH = 16
else:
    REC_BATCH = 8
print(f'Recommended batch size: {REC_BATCH}')

device = torch.device('cuda')
WORK_DIR = '/content/mivolo_finetune'
os.makedirs(WORK_DIR, exist_ok=True)
print('All dependencies installed.')

In [ ]:
#@title Step 2: Download & Prepare Indian Face Dataset
DATASET_DIR = f'{WORK_DIR}/dataset'
IMAGES_DIR = f'{DATASET_DIR}/images'
os.makedirs(IMAGES_DIR, exist_ok=True)

all_samples = []

# ═══════════════════════════════════════════════════════════
# Part A: Download UTKFace & filter Indian faces (race=3)
# ═══════════════════════════════════════════════════════════
UTKFACE_DIR = f'{WORK_DIR}/utkface_raw'
os.makedirs(UTKFACE_DIR, exist_ok=True)

if not glob.glob(f'{UTKFACE_DIR}/*.jpg'):
    print('Downloading UTKFace dataset...')
    try:
        import kagglehub
        path = kagglehub.dataset_download('jangedoo/utkface-new')
        print(f'UTKFace downloaded to: {path}')
        for f in Path(path).rglob('*.jpg'):
            shutil.copy2(str(f), UTKFACE_DIR)
        # Also check for chip.jpg files in subdirs
        for f in Path(path).rglob('*.chip.jpg'):
            shutil.copy2(str(f), UTKFACE_DIR)
    except Exception as e:
        print(f'kagglehub failed: {e}')
        print('Please upload UTKFace images manually to:', UTKFACE_DIR)
        print('Or set up Kaggle API key and rerun.')
else:
    print(f'UTKFace already downloaded ({len(glob.glob(f"{UTKFACE_DIR}/*.jpg"))} files)')

# Parse UTKFace: [age]_[gender]_[race]_[date].jpg, race=3 is Indian
utk_pattern = re.compile(r'^(\d+)_(\d)_(\d)_(\d+)')
gender_map = {'0': 'Male', '1': 'Female'}

for img_path in Path(UTKFACE_DIR).rglob('*.jpg'):
    match = utk_pattern.match(img_path.name)
    if not match:
        continue
    age = int(match.group(1))
    gender_code = match.group(2)
    race = int(match.group(3))
    if race != 3 or age < 1 or age > 100:
        continue
    dest_name = f'utk_{img_path.name}'
    dest_path = os.path.join(IMAGES_DIR, dest_name)
    if not os.path.exists(dest_path):
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        img = cv2.resize(img, (224, 224), interpolation=cv2.INTER_CUBIC)
        cv2.imwrite(dest_path, img, [cv2.IMWRITE_JPEG_QUALITY, 95])
    all_samples.append({
        'filename': dest_name, 'age': age,
        'gender': gender_map.get(gender_code, 'Unknown'), 'source': 'utkface'
    })

print(f'UTKFace Indian subset: {len(all_samples)} images')

# ═══════════════════════════════════════════════════════════
# Part B: Download IFAD (Indian Face Age Database)
# ═══════════════════════════════════════════════════════════
IFAD_DIR = f'{WORK_DIR}/ifad_raw'
if not os.path.isdir(IFAD_DIR):
    print('Cloning IFAD repository...')
    os.system(f'git clone --depth 1 https://github.com/IndianAgeDatabase/IFAD.git {IFAD_DIR}')
else:
    print('IFAD already cloned')

ifad_count = 0
if os.path.isdir(IFAD_DIR):
    for subject_dir in sorted(Path(IFAD_DIR).iterdir()):
        # Skip hidden dirs (.git), files, and non-subject dirs
        if not subject_dir.is_dir():
            continue
        if subject_dir.name.startswith(('.', '_', 'README')):
            continue
        for img_path in sorted(subject_dir.rglob('*')):
            if img_path.suffix.lower() not in ('.jpg', '.jpeg', '.png', '.bmp'):
                continue
            match = re.search(r'(\d{1,3})', img_path.stem)
            if not match:
                continue
            age = int(match.group(1))
            if age < 1 or age > 100:
                continue
            dest_name = f'ifad_{subject_dir.name}_{img_path.stem}.jpg'
            dest_path = os.path.join(IMAGES_DIR, dest_name)
            if not os.path.exists(dest_path):
                img = cv2.imread(str(img_path))
                if img is None:
                    continue
                # Upscale small IFAD images (128x128) to 224x224
                img = cv2.resize(img, (224, 224), interpolation=cv2.INTER_CUBIC)
                cv2.imwrite(dest_path, img, [cv2.IMWRITE_JPEG_QUALITY, 95])
            all_samples.append({
                'filename': dest_name, 'age': age,
                'gender': 'Unknown', 'source': 'ifad'
            })
            ifad_count += 1

print(f'IFAD: {ifad_count} images')
print(f'\nTotal dataset: {len(all_samples)} images')

if len(all_samples) == 0:
    raise ValueError('No samples collected! Check dataset downloads above.')

# Write unified CSV
CSV_PATH = f'{DATASET_DIR}/labels.csv'
with open(CSV_PATH, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['filename', 'age', 'gender', 'source'])
    writer.writeheader()
    writer.writerows(all_samples)

# Print age distribution
ages = [s['age'] for s in all_samples]
sources = {}
for s in all_samples:
    sources[s['source']] = sources.get(s['source'], 0) + 1
print(f'Age range: {min(ages)} - {max(ages)}, Mean: {sum(ages)/len(ages):.1f}')
print(f'Sources: {sources}')
print('\nAge Distribution:')
for decade in range(0, 110, 10):
    count = sum(1 for a in ages if decade <= a < decade + 10)
    bar = '#' * (count // max(1, len(ages) // 40)) or '|'
    print(f'  {decade:>3d}-{decade+9:<3d}: {count:>5d}  {bar}')

In [ ]:
#@title Step 3: Load MiVOLO v2 Pretrained Model
from transformers import AutoModelForImageClassification, AutoConfig, AutoImageProcessor

MODEL_NAME = 'iitolstykh/mivolo_v2'
print(f'Loading {MODEL_NAME}...')

model_config = AutoConfig.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForImageClassification.from_pretrained(
    MODEL_NAME, trust_remote_code=True, dtype=torch.float32,
)
model = model.to(device)

processor = AutoImageProcessor.from_pretrained(MODEL_NAME, trust_remote_code=True)

total_params = sum(p.numel() for p in model.parameters())
print(f'Model loaded: {total_params:,} parameters on {device}')

In [ ]:
#@title Step 4: Create Dataset & Evaluate Baseline

# ═══════════════════════════════════════════════════════════
# Dataset class
# ═══════════════════════════════════════════════════════════
class IndianFaceDataset(Dataset):
    GENDER_MAP = {'male': 0, 'man': 0, 'female': 1, 'woman': 1}

    def __init__(self, data_root, labels_csv, processor, augment=False):
        self.data_root = data_root
        self.processor = processor
        self.augment = augment
        self.aug_transform = T.Compose([
            T.RandomHorizontalFlip(0.5),
            T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.05),
            T.RandomAffine(degrees=10, translate=(0.05, 0.05), scale=(0.9, 1.1)),
        ]) if augment else None
        self.samples = []
        with open(labels_csv, 'r') as f:
            reader = csv.DictReader(f)
            for row in reader:
                path = os.path.join(data_root, row['filename'])
                if not os.path.isfile(path):
                    continue
                age = max(0.0, min(100.0, float(row['age'])))
                gender = self.GENDER_MAP.get(row.get('gender', '').strip().lower(), -1)
                self.samples.append({'path': path, 'age': age, 'gender': gender})
        print(f'  Dataset loaded: {len(self.samples)} samples (augment={augment})')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        img = Image.open(s['path']).convert('RGB')
        if self.aug_transform is not None:
            img = self.aug_transform(img)
        # MiVOLO processor requires List[numpy.ndarray], not PIL
        img_np = np.array(img)
        face = self.processor(images=[img_np])['pixel_values'][0]
        body = torch.zeros_like(face)
        return {
            'face_pixel_values': face,
            'body_pixel_values': body,
            'age': torch.tensor(s['age'], dtype=torch.float32),
            'gender': torch.tensor(s['gender'], dtype=torch.long),
        }

# ═══════════════════════════════════════════════════════════
# Train/Val split
# ═══════════════════════════════════════════════════════════
full_dataset = IndianFaceDataset(IMAGES_DIR, CSV_PATH, processor, augment=False)
total = len(full_dataset)
val_size = max(1, int(total * 0.1))
train_size = total - val_size

train_indices, val_indices = random_split(
    range(total), [train_size, val_size],
    generator=torch.Generator().manual_seed(42),
)

val_subset = Subset(full_dataset, val_indices.indices)
# num_workers=0 is critical for Colab — multiprocessing workers crash the kernel
val_loader = DataLoader(val_subset, batch_size=REC_BATCH, shuffle=False,
                        num_workers=0, pin_memory=True)

print(f'Split: {train_size} train / {val_size} val')

# ═══════════════════════════════════════════════════════════
# Baseline evaluation (before fine-tuning)
# ═══════════════════════════════════════════════════════════
print('\nEvaluating baseline (pretrained MiVOLO v2 on Indian faces)...')
model.eval()
total_mae = 0.0
n_samples = 0

with torch.no_grad():
    for batch in val_loader:
        face = batch['face_pixel_values'].to(device)
        body = batch['body_pixel_values'].to(device)
        age_gt = batch['age'].to(device)
        output = model(faces_input=face, body_input=body)
        pred = output.age_output.squeeze(-1)
        total_mae += (pred - age_gt).abs().sum().item()
        n_samples += age_gt.size(0)

baseline_mae = total_mae / max(n_samples, 1)
print(f'Baseline MAE on Indian faces: {baseline_mae:.2f} years ({n_samples} samples)')

In [ ]:
#@title Step 5: Configure & Start Training
from tqdm.auto import tqdm

# ═══════════════════════════════════════════════════════════
# Hyperparameters (edit these if needed)
# ═══════════════════════════════════════════════════════════
NUM_EPOCHS = 30           #@param {type:"integer"}
BATCH_SIZE = REC_BATCH    #@param {type:"integer"}
LEARNING_RATE = 2e-5      #@param {type:"number"}
FREEZE_EPOCHS = 5         #@param {type:"integer"}
UNFREEZE_BLOCKS = 4       #@param {type:"integer"}
AGE_LOSS = 'l1'           #@param ['l1', 'huber', 'mse']
PATIENCE = 7              #@param {type:"integer"}
OUTPUT_DIR = f'{WORK_DIR}/checkpoints'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ═══════════════════════════════════════════════════════════
# Freeze / Unfreeze helpers
# ═══════════════════════════════════════════════════════════
def freeze_all(m):
    for p in m.parameters():
        p.requires_grad = False

def unfreeze_heads(m):
    head_keys = ('head', 'classifier', 'age', 'gender', 'fc', 'output')
    for name, p in m.named_parameters():
        if any(k in name.lower() for k in head_keys):
            p.requires_grad = True

def unfreeze_last_blocks(m, n):
    block_params = []
    for name, param in m.named_parameters():
        if 'body' in name.lower():
            continue
        for pat in ('blocks.', 'layers.', 'encoder.layer.'):
            if pat in name:
                try:
                    idx = int(name.split(pat)[1].split('.')[0])
                    block_params.append((idx, name, param))
                except (ValueError, IndexError):
                    pass
                break
    if block_params:
        max_idx = max(bp[0] for bp in block_params)
        for idx, name, param in block_params:
            if idx >= max_idx - n + 1:
                param.requires_grad = True
    else:
        # Fallback: unfreeze all non-body params
        for name, p in m.named_parameters():
            if 'body' not in name.lower():
                p.requires_grad = True

def build_optimizer(m, lr):
    head_keys = ('head', 'classifier', 'age', 'gender', 'fc', 'output')
    head_p, backbone_p = [], []
    for name, p in m.named_parameters():
        if not p.requires_grad:
            continue
        if any(k in name.lower() for k in head_keys):
            head_p.append(p)
        else:
            backbone_p.append(p)
    groups = []
    if head_p:
        groups.append({'params': head_p, 'lr': lr * 5})
    if backbone_p:
        groups.append({'params': backbone_p, 'lr': lr})
    if not groups:
        groups = [{'params': m.parameters(), 'lr': lr}]
    return torch.optim.AdamW(groups, weight_decay=0.01)

# ═══════════════════════════════════════════════════════════
# Prepare data loaders
# num_workers=0 is critical for Colab stability
# ═══════════════════════════════════════════════════════════
train_dataset = IndianFaceDataset(IMAGES_DIR, CSV_PATH, processor, augment=True)
train_subset = Subset(train_dataset, train_indices.indices)
train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=0, pin_memory=True, drop_last=True)

# ═══════════════════════════════════════════════════════════
# Loss, optimizer, scaler
# ═══════════════════════════════════════════════════════════
if AGE_LOSS == 'l1':
    age_criterion = nn.L1Loss()
elif AGE_LOSS == 'huber':
    age_criterion = nn.SmoothL1Loss(beta=2.0)
else:
    age_criterion = nn.MSELoss()
gender_criterion = nn.CrossEntropyLoss(ignore_index=-1)

# Initial freeze: heads only
freeze_all(model)
unfreeze_heads(model)
optimizer = build_optimizer(model, LEARNING_RATE)
scaler = torch.amp.GradScaler('cuda')

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Warmup phase: {trainable:,} trainable params (heads only)')

# ═══════════════════════════════════════════════════════════
# Training loop
# ═══════════════════════════════════════════════════════════
best_mae = float('inf')
patience_counter = 0
history = {'train_mae': [], 'val_mae': [], 'train_loss': [], 'val_loss': []}

print('='*60)
print(f'  MiVOLO v2 Fine-tuning on Indian Faces')
print(f'  {len(train_subset)} train / {len(val_subset)} val')
print(f'  GPU: {torch.cuda.get_device_name(0)}')
print(f'  Epochs: {NUM_EPOCHS} | Batch: {BATCH_SIZE} | LR: {LEARNING_RATE}')
print(f'  Freeze backbone: {FREEZE_EPOCHS} epochs | Unfreeze blocks: {UNFREEZE_BLOCKS}')
print('='*60)

for epoch in range(1, NUM_EPOCHS + 1):
    t0 = time.time()

    # Unfreeze backbone after warmup
    if epoch == FREEZE_EPOCHS + 1:
        print(f'\n>>> Epoch {epoch}: Unfreezing last {UNFREEZE_BLOCKS} transformer blocks')
        unfreeze_last_blocks(model, UNFREEZE_BLOCKS)
        optimizer = build_optimizer(model, LEARNING_RATE)
        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f'    Trainable params: {trainable:,}\n')

    # ── Train ──
    model.train()
    train_loss, train_mae, n_train = 0.0, 0.0, 0
    pbar = tqdm(train_loader, desc=f'Epoch {epoch}/{NUM_EPOCHS}', leave=False)

    for batch in pbar:
        face = batch['face_pixel_values'].to(device)
        body = batch['body_pixel_values'].to(device)
        age_gt = batch['age'].to(device)
        gender_gt = batch['gender'].to(device)

        optimizer.zero_grad()

        with torch.amp.autocast('cuda'):
            out = model(faces_input=face, body_input=body)
            age_pred = out.age_output.squeeze(-1)
            loss = age_criterion(age_pred, age_gt)
            # Add gender loss for samples with known gender
            valid_g = gender_gt != -1
            if valid_g.any() and hasattr(out, 'gender_output'):
                loss = loss + 0.3 * gender_criterion(
                    out.gender_output[valid_g], gender_gt[valid_g]
                )

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()

        bs = age_gt.size(0)
        train_loss += loss.item() * bs
        train_mae += (age_pred.detach() - age_gt).abs().sum().item()
        n_train += bs
        pbar.set_postfix(mae=f'{train_mae/n_train:.2f}', loss=f'{train_loss/n_train:.4f}')

    # ── Validate ──
    model.eval()
    val_loss_total, val_mae_total, n_val = 0.0, 0.0, 0
    with torch.no_grad():
        for batch in val_loader:
            face = batch['face_pixel_values'].to(device)
            body = batch['body_pixel_values'].to(device)
            age_gt = batch['age'].to(device)
            out = model(faces_input=face, body_input=body)
            pred = out.age_output.squeeze(-1)
            val_loss_total += age_criterion(pred, age_gt).item() * age_gt.size(0)
            val_mae_total += (pred - age_gt).abs().sum().item()
            n_val += age_gt.size(0)

    ep_train_mae = train_mae / max(n_train, 1)
    ep_val_mae = val_mae_total / max(n_val, 1)
    ep_train_loss = train_loss / max(n_train, 1)
    ep_val_loss = val_loss_total / max(n_val, 1)
    elapsed = time.time() - t0

    history['train_mae'].append(ep_train_mae)
    history['val_mae'].append(ep_val_mae)
    history['train_loss'].append(ep_train_loss)
    history['val_loss'].append(ep_val_loss)

    improved = ''
    if ep_val_mae < best_mae:
        best_mae = ep_val_mae
        patience_counter = 0
        torch.save({
            'model_state_dict': model.state_dict(),
            'config': {'model_name': MODEL_NAME, 'age_loss': AGE_LOSS},
            'epoch': epoch,
            'val_mae': best_mae,
        }, f'{OUTPUT_DIR}/mivolo_indian_best.pt')
        improved = ' * BEST'
    else:
        patience_counter += 1

    print(f'Epoch {epoch:>2d}/{NUM_EPOCHS} [{elapsed:.0f}s] '
          f'Train MAE: {ep_train_mae:.2f} Loss: {ep_train_loss:.4f} | '
          f'Val MAE: {ep_val_mae:.2f} Loss: {ep_val_loss:.4f}{improved}')

    # Periodic checkpoint
    if epoch % 5 == 0:
        torch.save({
            'model_state_dict': model.state_dict(),
            'epoch': epoch, 'val_mae': ep_val_mae,
        }, f'{OUTPUT_DIR}/mivolo_indian_epoch{epoch}.pt')

    # Early stopping
    if PATIENCE > 0 and patience_counter >= PATIENCE:
        print(f'\nEarly stopping at epoch {epoch} (no improvement for {PATIENCE} epochs)')
        break

print(f'\n{"="*60}')
print(f'  TRAINING COMPLETE')
print(f'  Baseline MAE: {baseline_mae:.2f}')
print(f'  Best Val MAE: {best_mae:.2f}')
print(f'  Improvement:  {baseline_mae - best_mae:.2f} years')
print(f'{"="*60}')

In [ ]:
#@title Step 6: Training Curves
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
epochs_range = range(1, len(history['train_mae']) + 1)

ax1.plot(epochs_range, history['train_mae'], 'b-', label='Train MAE')
ax1.plot(epochs_range, history['val_mae'], 'r-', label='Val MAE')
ax1.axhline(y=baseline_mae, color='gray', linestyle='--', label=f'Baseline ({baseline_mae:.2f})')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('MAE (years)')
ax1.set_title('Age Estimation MAE')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(epochs_range, history['train_loss'], 'b-', label='Train Loss')
ax2.plot(epochs_range, history['val_loss'], 'r-', label='Val Loss')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.set_title('Training Loss')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/training_curves.png', dpi=150)
plt.show()
print(f'Saved: {OUTPUT_DIR}/training_curves.png')

In [ ]:
#@title Step 7: Evaluate Fine-tuned Model (Per Age Group)
# Load best checkpoint
best_ckpt = torch.load(f'{OUTPUT_DIR}/mivolo_indian_best.pt',
                        map_location=device, weights_only=False)
model.load_state_dict(best_ckpt['model_state_dict'])
model.eval()
print(f'Loaded best checkpoint (epoch {best_ckpt["epoch"]}, val MAE {best_ckpt["val_mae"]:.2f})')

# Per-age-group evaluation
age_groups = {'0-20': (0, 20), '21-35': (21, 35), '36-50': (36, 50),
              '51-70': (51, 70), '71+': (71, 100)}
group_errors = {k: [] for k in age_groups}

with torch.no_grad():
    for batch in val_loader:
        face = batch['face_pixel_values'].to(device)
        body = batch['body_pixel_values'].to(device)
        age_gt = batch['age']
        pred = model(faces_input=face, body_input=body).age_output.squeeze(-1).cpu()
        for gt, p in zip(age_gt, pred):
            err = abs(p.item() - gt.item())
            for group, (lo, hi) in age_groups.items():
                if lo <= gt.item() <= hi:
                    group_errors[group].append(err)
                    break

print(f'\n{"Age Group":>10s}  {"Count":>6s}  {"MAE":>6s}')
print('-' * 30)
for group, errors in group_errors.items():
    if errors:
        print(f'{group:>10s}  {len(errors):>6d}  {sum(errors)/len(errors):>6.2f}')
    else:
        print(f'{group:>10s}  {0:>6d}  {"N/A":>6s}')

all_errors = [e for errs in group_errors.values() for e in errs]
if all_errors:
    print(f'{"Overall":>10s}  {len(all_errors):>6d}  {sum(all_errors)/len(all_errors):>6.2f}')
    print(f'\nBaseline was {baseline_mae:.2f} -> Now {sum(all_errors)/len(all_errors):.2f}')

In [ ]:
#@title Step 8: Save to Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

DRIVE_SAVE_DIR = '/content/drive/MyDrive/AgeVision/checkpoints'
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)

best_path = f'{OUTPUT_DIR}/mivolo_indian_best.pt'
if os.path.isfile(best_path):
    dest = f'{DRIVE_SAVE_DIR}/mivolo_indian_best.pt'
    print(f'Saving to: {dest}')
    shutil.copy2(best_path, dest)
    size_mb = os.path.getsize(dest) / 1e6
    print(f'Saved! ({size_mb:.1f} MB)')

    curves = f'{OUTPUT_DIR}/training_curves.png'
    if os.path.isfile(curves):
        shutil.copy2(curves, f'{DRIVE_SAVE_DIR}/mivolo_training_curves.png')

    print(f'\n--- NEXT STEPS ---')
    print(f'1. Download mivolo_indian_best.pt from Google Drive')
    print(f'2. Copy to: agevision_backend/checkpoints/mivolo_indian_best.pt')
    print(f'3. Restart Django server - it will auto-detect the fine-tuned model')
    print(f'4. Baseline MAE: {baseline_mae:.2f} -> Fine-tuned MAE: {best_mae:.2f}')
else:
    print('No checkpoint found to save')

In [ ]:
#@title (Optional) Download directly to your computer
from google.colab import files

best_path = f'{OUTPUT_DIR}/mivolo_indian_best.pt'
if os.path.isfile(best_path):
    size_mb = os.path.getsize(best_path) / 1e6
    print(f'Downloading mivolo_indian_best.pt ({size_mb:.1f} MB)...')
    files.download(best_path)
else:
    print('No checkpoint to download')